# 04 — Model C: recommended loan amount

Dataset: **Approved** rows of `loan_approval_dataset.csv` only  
Target: `loan_amount` (right-skewed → `log1p` via `TransformedTargetRegressor`)

`loan_amount` is **not** a feature (no leakage). At inference the API must return `min(requested_amount, prediction)`.


In [ ]:
%matplotlib inline


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "ml" / "pipeline" / "features.py").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.config import (
    DATA_RAW,
    FIGURES_DIR,
    MODELS_DIR,
    RANDOM_STATE,
    REPORTS_DIR,
    ensure_dirs,
)

ensure_dirs()
print("Project root:", ROOT)


In [ ]:
import json
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")


def ks_statistic(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    pos, neg = y_score[y_true == 1], y_score[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    return float(ks_2samp(pos, neg).statistic)


def classification_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (np.asarray(y_proba) >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "ks_statistic": ks_statistic(y_true, y_proba),
        "average_precision": float(average_precision_score(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(report["1"]["precision"]),
        "recall": float(report["1"]["recall"]),
        "classification_report": report,
    }


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": float(np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1, None))) * 100),
    }


def stratified_split(X, y, random_state=RANDOM_STATE):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def save_pipeline(pipe, name, metadata):
    path = MODELS_DIR / f"{name}.pkl"
    joblib.dump(pipe, path)
    metadata = {**metadata, "artifact": str(path.as_posix()), "saved_at": datetime.now(timezone.utc).isoformat()}
    (MODELS_DIR / f"{name}.meta.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
    print("Saved", path)
    return path


In [ ]:
from sklearn import set_config
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from ml.pipeline.features import LoanAmountFeatures, strip_loan_frame

set_config(transform_output="pandas")

csv = DATA_RAW / "loan_approval_dataset.csv"
if not csv.exists():
    csv = ROOT / "loan_approval_dataset.csv"
loan_df = strip_loan_frame(pd.read_csv(csv))
work = loan_df[loan_df["loan_status"] == "Approved"].reset_index(drop=True)
y = work["loan_amount"].astype(float)
X = work.drop(columns=[c for c in ("loan_status", "loan_id", "loan_amount") if c in work.columns])

strata = pd.qcut(y, q=5, duplicates="drop")
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=strata, random_state=RANDOM_STATE)
temp_strata = pd.qcut(y_temp, q=min(5, y_temp.nunique()), duplicates="drop")
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=temp_strata, random_state=RANDOM_STATE)
print(f"train/val/test = {len(X_train)}/{len(X_val)}/{len(X_test)}  target median={float(y_train.median()):,.0f}")


In [ ]:
CAT_COLS = ["education", "self_employed", "risk_tier"]
preview = LoanAmountFeatures().fit(X_train).transform(X_train.head(5))
assert "loan_amount" not in preview.columns
cat = [c for c in CAT_COLS if c in preview.columns]
num = [c for c in preview.columns if c not in cat]

pipe = Pipeline([
    ("features", LoanAmountFeatures()),
    ("preprocess", ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat), ("num", "passthrough", num)],
        remainder="drop",
        verbose_feature_names_out=False,
    )),
    ("model", TransformedTargetRegressor(
        regressor=XGBRegressor(
            n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
            min_child_weight=4, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
        ),
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )),
])
pipe.fit(X_train, y_train)
val_pred, test_pred = pipe.predict(X_val), pipe.predict(X_test)
val_metrics, test_metrics = regression_metrics(y_val, val_pred), regression_metrics(y_test, test_pred)
print(f"Val  MAE={val_metrics['mae']:,.0f}  RMSE={val_metrics['rmse']:,.0f}  MAPE={val_metrics['mape']:.1f}%")
print(f"Test MAE={test_metrics['mae']:,.0f}  RMSE={test_metrics['rmse']:,.0f}  MAPE={test_metrics['mape']:.1f}%")


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, test_pred, alpha=0.25, s=12)
lo, hi = min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())
plt.plot([lo, hi], [lo, hi], "--", color="gray")
plt.xlabel("Actual loan amount")
plt.ylabel("Predicted loan amount")
plt.title("Model C — recommended amount")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "model_c_pred_vs_actual.png", dpi=140)
plt.show()


In [ ]:
import shap

transformed = pipe.named_steps["preprocess"].transform(pipe.named_steps["features"].transform(X_test))
sample = transformed.sample(n=min(200, len(transformed)), random_state=1)
inner = pipe.named_steps["model"].regressor_
shap.summary_plot(shap.TreeExplainer(inner).shap_values(sample), sample, show=False, max_display=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_summary_model_c.png", dpi=140, bbox_inches="tight")
plt.show()


## Inference sanity rule (belongs in the API, shown here)


In [ ]:
requested = 20_000_000
predicted = float(pipe.predict(X_test.head(1))[0])
recommended = min(requested, predicted)
print(f"predicted={predicted:,.0f}  requested={requested:,.0f}  recommended={recommended:,.0f}  capped={recommended < predicted}")


In [ ]:
save_pipeline(
    pipe,
    "model_c_amount",
    {
        "model_version": "model_c_v1",
        "task": "recommended_loan_amount",
        "target_transform": "log1p / expm1",
        "metrics": {"val": val_metrics, "test": test_metrics},
        "inference_rule": "recommended_amount = min(requested_amount, model_prediction)",
    },
)
loaded = joblib.load(MODELS_DIR / "model_c_amount.pkl")
print("sample recommended amount:", float(loaded.predict(X_test.head(1))[0]))
